# Zipline Reloaded Orchestrator Tutorial

This notebook shows how to use `quant-orchestrator` with `quant-warehouse` for multi-vendor, single-symbol and multi-symbol research, Monte Carlo simulation, walk-forward optimization, equity-curve analysis, and portfolio optimization.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.experiments import WindowSpec
from quant_orchestrator.monte_carlo import simulate_return_paths
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    build_sma_crossover_frame,
    load_price_frame,
    load_signal_frame,
    normalize_session_label,
)
from quant_orchestrator.strategy import summarize_backtest, summarize_equity

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

SINGLE_SYMBOL = 'AAPL'
MULTI_VENDOR_PROVIDERS = ('fmp', 'yfinance')
TRAIN_START = '2020-01-01'
TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'
TEST_END = None
FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    frame = build_sma_crossover_frame(prices, fast_window=fast_window, slow_window=slow_window)
    frame.attrs['fast_window'] = fast_window
    frame.attrs['slow_window'] = slow_window
    return frame


def combine_equity_sleeves(sleeves: list[pd.Series]) -> pd.Series:
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index, name='portfolio_value')
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index).ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index()


def align_return_frame(equity_map: dict[str, pd.Series]) -> pd.DataFrame:
    return pd.concat(
        {name: equity.pct_change().fillna(0.0) for name, equity in equity_map.items()},
        axis=1,
    ).fillna(0.0)


def max_sharpe_weights(return_frame: pd.DataFrame) -> pd.Series:
    if return_frame.empty:
        raise ValueError('return_frame cannot be empty')
    columns = list(return_frame.columns)
    mean = return_frame.mean().to_numpy(dtype=float)
    cov = return_frame.cov().to_numpy(dtype=float)
    if len(columns) == 1:
        return pd.Series([1.0], index=columns)

    def neg_sharpe(weights: np.ndarray) -> float:
        portfolio_mean = float(weights @ mean)
        portfolio_vol = float(np.sqrt(weights @ cov @ weights))
        if portfolio_vol <= 0:
            return 1e9
        return -(portfolio_mean / portfolio_vol)

    bounds = [(0.0, 1.0)] * len(columns)
    constraints = ({'type': 'eq', 'fun': lambda weights: float(np.sum(weights)) - 1.0},)
    guess = np.repeat(1.0 / len(columns), len(columns))
    result = minimize(neg_sharpe, guess, bounds=bounds, constraints=constraints, method='SLSQP')
    weights = pd.Series(result.x if result.success else guess, index=columns)
    weights = weights.clip(lower=0.0)
    return weights / weights.sum()


def weighted_equity_from_returns(return_frame: pd.DataFrame, weights: pd.Series, *, capital_base: float) -> pd.Series:
    aligned = return_frame.loc[:, weights.index].fillna(0.0)
    portfolio_returns = aligned.mul(weights, axis=1).sum(axis=1)
    equity = capital_base * (1.0 + portfolio_returns).cumprod()
    return equity.rename('portfolio_value')


def wfo_windows() -> WindowSpec:
    return WindowSpec(
        mode='fixed',
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        test_start=TEST_START,
        test_end=TEST_END,
    )


def run_monte_carlo(equity: pd.Series, *, iterations: int = 1000, block_size: int = 5) -> pd.DataFrame:
    returns = equity.pct_change().dropna()
    mc = simulate_return_paths(returns, iterations=iterations, block_size=block_size, horizon=len(returns))
    return mc.summary


def summarize_candidates(rows: list[dict[str, object]]) -> pd.DataFrame:
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    return table.sort_values(by=['total_return', 'max_drawdown', 'final_equity'], ascending=[False, True, False]).reset_index(drop=True)

FRAMEWORK_TITLE = 'zipline'
from quant_orchestrator.platforms.backtesting_frameworks.zipline.sma_crossover import (
    run_sma_crossover_backtest as run_zipline_sma_crossover_backtest,
)


def run_framework_backtest(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    fast_window = int(frame.attrs.get('fast_window', 50))
    slow_window = int(frame.attrs.get('slow_window', 200))
    prices = frame.loc[:, ['open', 'high', 'low', 'close', 'volume']].copy()
    return run_zipline_sma_crossover_backtest(
        prices,
        symbol=symbol,
        fast_window=fast_window,
        slow_window=slow_window,
        capital_base=capital_base,
    )


## Single Symbol / Multi Vendor


In [2]:

print(f'Tutorial framework: {FRAMEWORK_TITLE}')
print(f'Single symbol: {SINGLE_SYMBOL}')
print(f'Vendors: {MULTI_VENDOR_PROVIDERS}')


Tutorial framework: zipline
Single symbol: AAPL
Vendors: ('fmp', 'yfinance')


## Multi Vendor Backtest


In [3]:

def run_multi_vendor_single_symbol(symbol: str = SINGLE_SYMBOL, providers: tuple[str, ...] = MULTI_VENDOR_PROVIDERS, capital_base: float = 100_000.0) -> pd.DataFrame:
    rows = []
    for provider in providers:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        rows.append(summary.assign(provider=provider))
    return pd.concat(rows, ignore_index=True)

vendor_report = run_multi_vendor_single_symbol()
display(vendor_report.round(4).head(10))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,fast_window,slow_window,native_last_value,provider
0,zipline,AAPL,2021-08-02,2026-06-24,1229,9,105274.06,0.0527,-0.2152,0.0075,0.6979,1760.99,50,200,105274.0618,fmp
1,zipline,AAPL,2021-08-02,2026-06-23,1228,9,107639.03,0.0764,-0.2132,0.0074,0.4084,3006.86,50,200,107639.0345,yfinance


## Multi Symbol Backtest


In [4]:

def run_multi_symbol_backtest(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    symbol_rows = []
    sleeves = []
    runs = {}
    elapsed = 0.0
    trades = 0
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        started = perf_counter()
        raw, summary, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        elapsed += perf_counter() - started
        symbol_rows.append(summary.assign(provider=provider))
        sleeves.append(equity.rename(symbol))
        trades += int(summary['trades'].iloc[0])
        runs[symbol] = raw
    portfolio_equity = combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol='MAG7',
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary['provider'] = provider
    return pd.concat(symbol_rows, ignore_index=True), portfolio_summary, runs, portfolio_equity

symbol_report, portfolio_report, runs_by_symbol, portfolio_equity = run_multi_symbol_backtest()
print('Symbol report:')
display(symbol_report.loc[:, ['provider', 'symbol', 'start', 'end', 'bars', 'trades', 'final_equity', 'total_return', 'max_drawdown', 'elapsed_seconds']].round(4).head(10))
print()
print('Portfolio report:')
display(portfolio_report.round(4).head(10))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


Symbol report:


,provider,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,AAPL,2021-08-02,2026-06-24,1229,9,15028.54,0.0520,-0.2126,0.3940
1,fmp,AMZN,2021-08-02,2026-06-24,1229,7,13316.34,-0.0679,-0.2823,0.4637
2,fmp,GOOGL,2021-08-02,2026-06-24,1229,5,20565.16,0.4396,-0.1103,0.4113
3,fmp,META,2021-08-02,2026-06-24,1229,6,19867.72,0.3907,-0.1766,0.4212
4,fmp,MSFT,2021-08-02,2026-06-24,1229,10,15878.32,0.1115,-0.1697,0.3892
5,fmp,NVDA,2021-08-02,2026-06-24,1229,5,21822.90,0.5276,-0.1075,0.4616
6,fmp,TSLA,2021-08-02,2026-06-24,1229,10,12004.29,-0.1597,-0.4005,0.3999



Portfolio report:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,provider
0,zipline,MAG7,2021-08-02,2026-06-24,1229,52,118483.26,0.1848,-0.1386,0.0054,2.9559,415.77,fmp


## Monte Carlo


In [5]:

mc_report = run_monte_carlo(portfolio_equity, iterations=1000, block_size=5)
display(mc_report.round(4).head(10))


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1228,0.204,-0.1084,0.1914,0.5804,-0.1449,-0.2419


## Walk Forward Optimization


In [6]:

def run_walk_forward_optimization(symbol: str = SINGLE_SYMBOL, provider: str = 'fmp', capital_base: float = 100_000.0):
    spec = wfo_windows()
    train_frame = load_price_frame(symbol, provider=provider, start=spec.train_start, end=spec.train_end)
    test_frame = load_price_frame(symbol, provider=provider, start=spec.test_start, end=spec.test_end)

    train_rows = []
    for fast_window, slow_window in product(FAST_WINDOWS, SLOW_WINDOWS):
        if fast_window >= slow_window:
            continue
        frame = build_sma_frame(train_frame, fast_window=fast_window, slow_window=slow_window)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        row = summary.iloc[0].to_dict()
        row['fast_window'] = fast_window
        row['slow_window'] = slow_window
        train_rows.append(row)

    train_grid = summarize_candidates(train_rows)
    best = train_grid.iloc[0]
    best_fast = int(best['fast_window'])
    best_slow = int(best['slow_window'])
    _, _, test_equity = run_framework_backtest(build_sma_frame(test_frame, fast_window=best_fast, slow_window=best_slow), symbol=symbol, capital_base=capital_base)
    test_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol=symbol,
        equity=test_equity,
        elapsed_seconds=0.0,
        bars=len(test_frame),
        trades=None,
    )
    return train_grid, best, test_summary

train_grid, best_row, test_summary = run_walk_forward_optimization()
print('Train grid:')
display(train_grid.loc[:, ['fast_window', 'slow_window', 'final_equity', 'total_return', 'max_drawdown']].round(4).head(10))
print()
print('Best parameters:')
display(best_row.loc[['fast_window', 'slow_window', 'total_return', 'max_drawdown']].to_frame(name='value').round(4))
print()
print('Test summary:')
display(test_summary.round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

,fast_window,slow_window,final_equity,total_return,max_drawdown
0,10,200,140558.79,0.4056,-0.1240
1,50,100,140089.03,0.4009,-0.1300
2,20,100,139879.65,0.3988,-0.1719
3,10,100,130454.57,0.3045,-0.1811
4,20,200,129828.45,0.2983,-0.1739
5,10,150,129371.08,0.2937,-0.1419
6,20,150,124038.28,0.2404,-0.1580
7,50,200,123973.24,0.2397,-0.1379
8,50,150,121443.87,0.2144,-0.1433



Best parameters:


,value
fast_window,10
slow_window,200
total_return,0.4056
max_drawdown,-0.124



Test summary:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second
0,zipline,AAPL,2025-10-20,2026-06-24,369,None,108780.71,0.0878,-0.1049,0.0101,0.0,None


## Equity Curve Analysis


In [7]:

summary_view = pd.Series(summarize_equity(portfolio_equity), name='value').to_frame()
display(summary_view.round(4))
display(portfolio_equity.tail().to_frame(name='portfolio_value').round(2))


,value
start,2021-08-02
end,2026-06-24
final_equity,118483.26
total_return,0.1848
max_drawdown,-0.1386
daily_vol,0.0054


,portfolio_value
2026-06-17 20:00:00+00:00,119625.24
2026-06-18 20:00:00+00:00,120482.18
2026-06-22 20:00:00+00:00,119229.57
2026-06-23 20:00:00+00:00,118606.90
2026-06-24 20:00:00+00:00,118483.26


## Portfolio Optimization


In [8]:

def optimize_symbol_portfolio(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    equity_map = {}
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, _, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        equity_map[symbol] = equity
    return equity_map

symbol_equities = optimize_symbol_portfolio()
symbol_returns = align_return_frame(symbol_equities)
symbol_weights = max_sharpe_weights(symbol_returns)
symbol_portfolio_equity = weighted_equity_from_returns(symbol_returns, symbol_weights, capital_base=100_000.0)
print('Symbol-level weights:')
display(symbol_weights.sort_values(ascending=False).to_frame(name='weight').round(4))
print()
print('Symbol-level portfolio summary:')
display(pd.Series(summarize_equity(symbol_portfolio_equity), name='value').to_frame().round(4))

candidate_pairs = [(fast, slow) for fast in FAST_WINDOWS for slow in SLOW_WINDOWS if fast < slow]
strategy_equities = {}
strategy_rows = []
strategy_frame = load_price_frame(SINGLE_SYMBOL, provider='fmp', start=TRAIN_START, end=TEST_END)
for fast_window, slow_window in candidate_pairs:
    candidate_frame = build_sma_frame(strategy_frame, fast_window=fast_window, slow_window=slow_window)
    _, summary, equity = run_framework_backtest(candidate_frame, symbol=SINGLE_SYMBOL, capital_base=100_000.0)
    key = f'{fast_window}/{slow_window}'
    strategy_equities[key] = equity
    strategy_rows.append({
        'strategy': key,
        'fast_window': fast_window,
        'slow_window': slow_window,
        'total_return': float(summary['total_return'].iloc[0]),
        'max_drawdown': float(summary['max_drawdown'].iloc[0]),
    })
strategy_returns = align_return_frame(strategy_equities)
strategy_weights = max_sharpe_weights(strategy_returns)
strategy_portfolio_equity = weighted_equity_from_returns(strategy_returns, strategy_weights, capital_base=100_000.0)
print()
print('Strategy-level weights:')
display(strategy_weights.sort_values(ascending=False).head(10).to_frame(name='weight').round(4))
print()
print('Strategy-level portfolio summary:')
display(pd.Series(summarize_equity(strategy_portfolio_equity), name='value').to_frame().round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(



Strategy-level weights:


,weight
10/200,0.7468
50/100,0.2205
20/100,0.0327
20/150,0.0000
10/100,0.0000
10/150,0.0000
20/200,0.0000
50/150,0.0000
50/200,0.0000



Strategy-level portfolio summary:


,value
start,2020-05-26
end,2026-06-24
final_equity,138748.79
total_return,0.3875
max_drawdown,-0.1056
daily_vol,0.0055
